# ML-10 — Content Action Playbook

The validated output of Week 5/6 becomes a human-reviewed action plan here. Every row in the queue is a page; the human decides, the queue only orders and explains.

Lens: `skills/README.md` -> `skills/writing-honest-claims/SKILL.md` (claim ladder: observed -> directional -> decision-support, never causal) + `skills/flyrank/flyrank-data/SKILL.md` (data contract). All claims below use the words the evidence can carry. This playbook is **practical and non-production**: it is a review-order aid over one anonymized snapshot, not an automated refresh system.

## 1. Ranked actions + reason codes

**What this is.** A ranked queue of pages to *review* (not to auto-edit). The queue is ordered by the validated gradient-boosting probability from Week 5/6 (computed on the **6,163 client-grouped holdout pages** — pages whose clients were never in training). Each page gets an **archetype** + **reason code** + one **action** from a deterministic rule, so a human can see *why* a page is on the list in one line.

### Archetype -> action mapping (deterministic, in priority order)

| # | Archetype | Trigger (measured on the page) | Reason code | Action | Cost tier |
|---|---|---|---|---|---|
| 1 | giant | impressions >= 3,000 | `huge_reach` / `built_recently` | review and hand-check the numbers first (even a tiny trend shift is a big absolute move) | 1 (cheap check) |
| 2 | mature_stale | impressions >= 300, last update >= 180 days, age >= 270 days | `old_untouched_still_earning` | full content refresh (facts, structure, intent) | 3 (expensive) |
| 3 | visible_low_ctr | position 1-10, CTR < 0.5%, impressions >= 300 | `top10_but_low_ctr` | rewrite title/meta + one snippet test | 2 (medium) |
| 4 | visible_steady | position 1-10, impressions >= 300, not the above | `top10_needs_intent_check` | verify keyword-intent match before touching | 2 (medium) |
| 5 | quiet_low_reach | impressions < 300 | `below_noise_floor` | monitor only; do not act | 0 (none) |
| 6 | watch | anything else that ranked | `flagged_recheck_next_cycle` | recheck next cycle | 0 (none) |

Why these archetypes, in two lines: **the decay signal in this snapshot is mostly "old and untouched but still earning reach" (with `days_since_last_update` clumped on batch dates) or "visible but failing to convert clicks" (Week-4 signal CONFIRMED); the expensive miss is the genuinely big page the score under-ranks** (Week 6 error analysis: two ~500K-impression giants were near misses), so giants get rule #1 regardless of the score.

**The decay/refresh insight, in safe language.** *Observed* in this snapshot: pages whose trend direction is "down" skew older (median `content_age_days` 445 vs 391 for the rest) and are over-represented among already-visible pages with CTR under 0.5%. A score, not a cause: refreshing a page was not measured to *cause* recovery here — there is no matched experiment in a single snapshot. The queue below is **decision support for review order**, and the human's check (Section 3) is where the actual decision happens.

In [1]:
# --- 1. exact Week-5/6 pipeline: prep, split, train (nothing new to invent) ----
import json
import os
import shutil
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
SEED = 42

ROOT = Path(os.getcwd()).resolve()
for _ in range(6):
    if (ROOT / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        break
    ROOT = ROOT.parent
OUT = ROOT / "work" / "outputs"
FIG = ROOT / "work" / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

NUM_COUNT = ["search_volume", "cpc", "word_count", "char_count",
             "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
             "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d"]
NUM_RAW = ["competition", "days_with_impressions", "days_with_sessions",
           "content_age_days", "days_since_last_update", "ctr", "avg_position",
           "engagement_rate", "scroll_rate", "ai_traffic_pct"]
TIERS = ["competition_level", "age_tier", "freshness_tier",
         "word_count_tier", "impression_tier", "position_tier"]
NOMINAL = ["content_type", "main_intent"]

X = pd.DataFrame(index=df.index)
for c in NUM_COUNT:
    X["log_" + c] = np.log1p(pd.to_numeric(df[c], errors="coerce")).fillna(0).astype(float)
for c in NUM_RAW:
    X[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(float)
for c in TIERS:
    X[c] = df[c].astype("category").cat.codes.astype(int)
for c in NOMINAL:
    X = X.join(pd.get_dummies(df[c].fillna("unknown"), prefix=c, dtype=int).astype(int))
y = df["is_declining_label"].astype(int)

def week4_rule_score(imp, dsul, pos, ctr):
    stale = ((dsul >= 180) & (imp >= 300)).astype(int)
    gap = ((pos > 0) & (pos <= 10) & (ctr < 0.5) & (imp >= 300)).astype(int)
    return np.log1p(imp) * (1 + stale) * (1 + gap)

def precision_at_k(y_true, score, ks=(10, 20, 50, 100)):
    order = np.argsort(-np.asarray(score))
    yy = np.asarray(y_true)
    return [round(float(yy[order[:k]].mean()), 3) for k in ks]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tr_g, te_g = next(gss.split(X, y, groups=df["client_id"]))
Xtr, Xte, ytr, yte = X.iloc[tr_g], X.iloc[te_g], y.iloc[tr_g], y.iloc[te_g]

hgb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=5, random_state=SEED)
hgb.fit(Xtr, ytr)
pte = hgb.predict_proba(Xte)[:, 1]
rule_te = week4_rule_score(df["impressions_90d"].iloc[te_g].values, df["days_since_last_update"].iloc[te_g].values,
                           df["avg_position"].iloc[te_g].values, df["ctr"].iloc[te_g].values)
print(f"validated pipeline reproduced: test n={len(yte):,}  base rate={yte.mean():.3f}  "
      f"HGB P@10={precision_at_k(yte, pte)[0]:.2f}  rule P@10={precision_at_k(yte, rule_te)[0]:.2f}  "
      f"HGB AUC={roc_auc_score(yte, pte):.3f}")

validated pipeline reproduced: test n=6,163  base rate=0.511  HGB P@10=0.90  rule P@10=0.40  HGB AUC=0.622


In [2]:
# --- 2. ranked queue on the held-out test pages (validated set only) ---------
test = df.iloc[te_g].copy().reset_index(drop=True)
test["proba"] = pte
order = np.argsort(-test["proba"].values)
test = test.iloc[order].reset_index(drop=True)
test["rank"] = np.arange(1, len(test) + 1)
test["predicted"] = (test["proba"] >= 0.5).astype(int)

def archetype_of(imp, dsul, pos, ctr, age):
    visible = (pos > 0) & (pos <= 10)
    if imp >= 3000:
        if dsul <= 30:
            return "fresh_giant", "huge reach, built/updated recently - hand-check the numbers before any edit"
        return "giant", "huge reach - reviewing this page first costs least and protects the most"
    if (dsul >= 180) & (imp >= 300) & (age >= 270):
        return "mature_stale", "old, untouched, still earning reach - full refresh candidate"
    if visible & (ctr < 0.5) & (imp >= 300):
        return "visible_low_ctr", "top-10 but under 0.5% CTR - title/meta snippet refresh candidate"
    if visible & (imp >= 300):
        return "visible_steady", "top-10 with usable CTR - verify intent match before touching"
    if imp < 300:
        return "quiet_low_reach", "below the 300-impression noise floor - monitor, do not act"
    return "watch", "flagged but below action threshold - recheck next cycle"

ACTION = {
    "fresh_giant":        ("review_first_hand_check", 1),
    "giant":              ("review_first_hand_check", 1),
    "mature_stale":       ("full_content_refresh",    3),
    "visible_low_ctr":    ("rewrite_title_meta",      2),
    "visible_steady":     ("verify_intent_then_refresh", 2),
    "quiet_low_reach":    ("monitor_no_action",       0),
    "watch":              ("recheck_next_cycle",      0),
}

rows = []
for _, r in test.iterrows():
    tag, reason = archetype_of(r["impressions_90d"], r["days_since_last_update"],
                               r["avg_position"], r["ctr"], r["content_age_days"])
    action, cost = ACTION[tag]
    rows.append((tag, action, cost, reason))
arch = pd.DataFrame(rows, columns=["archetype", "action", "cost_tier", "reason"])

queue = pd.concat([test.reset_index(drop=True), arch], axis=1)
queue["reach_at_risk_k"] = (queue["impressions_90d"] / 1000).round(1)

cols_show = ["rank", "proba", "predicted", "archetype", "action", "cost_tier",
             "impressions_90d", "avg_position", "ctr", "days_since_last_update",
             "content_age_days", "main_intent", "content_type"]
print("=== TOP 12 of the queue (review the giants first) ===")
print(queue.head(12)[cols_show].to_string(index=False))

print("\n=== archetype mix across the whole held-out queue ===")
print(queue["archetype"].value_counts().to_string())

print("\n=== inside the flagged set (predicted=1): action mix ===")
print(queue.loc[queue["predicted"] == 1, "action"].value_counts().to_string())

=== TOP 12 of the queue (review the giants first) ===
 rank    proba  predicted       archetype                  action  cost_tier  impressions_90d  avg_position  ctr  days_since_last_update  content_age_days   main_intent    content_type
    1 0.972806          1 visible_low_ctr      rewrite_title_meta          2             2846           2.0 0.14                     106               106 informational keyword article
    2 0.969405          1 visible_low_ctr      rewrite_title_meta          2             1620           1.1 0.06                     106               106 transactional keyword article
    3 0.959948          1 visible_low_ctr      rewrite_title_meta          2              502           2.0 0.00                      20               147 transactional keyword article
    4 0.955241          1     fresh_giant review_first_hand_check          1             3568           1.9 0.42                      20                95 informational keyword article
    5 0.955015       

## 2. Intended use and limits

**Intended use.** An editor / content reviewer, facing one 90-day snapshots of search performance, reads the queue from the top:

1. The **giant** rows first — hand-verify the underlying numbers before any edit.
2. The **visible_low_ctr** rows — cheapest real win, title/meta work.
3. The **mature_stale** rows — durable but expensive; queue them by expected reach at risk.
4. Everything below: monitor.

The queue is a *review-order* aid. The probability column is a ranking score validated on held-out clients — it is **not** a forecast of next-quarter impressions.

**Where it stops being valid (the limits):**

- One anonymized snapshot = one moment in time. No temporal generalization is claimed.
- The label is *already-observed* trend direction — "flagged" means "matches the profile of pages observed declining", not "will decline".
- Validation covers held-out *clients* within this 32-client portfolio only. A brand-new client needs its own re-validation before the queue is trusted.
- P@10 = 0.90 on the held-out queue is a ranking metric in this sample; it is not a guarantee on the next export.
- Pseudonyms only — no client-level, keyword, or URL claims are made.

**Cost / value thinking.** Action effort grows left to right: a title/meta rewrite is cheap, a full refresh is expensive, and a giant first needs a hand-check that is nearly free. On the value side, `impressions_90d` is the reach at risk (a page the editor does not act on that keeps decaying). The summary below shows where the at-risk reach actually sits in the flagged set, so the editor spends effort where the reach is. Value is a *proxy* — impressions are what is at stake, not promised traffic after an edit.

One number worth reading twice: the top-50 rows by model probability cover only ~2% of the flagged at-risk reach, because the very biggest pages sit at *low* probability ranks (they are the Week-6 false-negative class, e.g. ~500K-impression freshened pages). That measured gap is exactly why rule #1 of this playbook forces a human hand-check of **giants first** regardless of the score: the cheap, highest-reach review is the one the probability ordering alone would bury.

In [3]:
# --- 3. cost / value: where the at-risk reach actually sits ------------------
flag = queue[queue["predicted"] == 1].copy()
val = (
    flag.groupby("archetype")
    .agg(n=("content_id", "count"),
         at_risk_reach=("impressions_90d", "sum"),
         avg_reach=("impressions_90d", "mean"),
         avg_cost_tier=("cost_tier", "mean"))
    .sort_values("at_risk_reach", ascending=False)
)
val["at_risk_reach"] = (val["at_risk_reach"] / 1e6).round(1)
val["avg_reach"] = val["avg_reach"].round(0)
print("flagged pages by archetype (valued by reach at risk, in millions):")
print(val.to_string())
total_reach_flag = int(flag["impressions_90d"].sum())
top50_reach = int(queue.head(50)["impressions_90d"].sum())
print(f"\nat-risk reach in the flagged set: {total_reach_flag:,} impressions")
print(f"reach covered by reviewing the top-50 rows: {top50_reach:,} impressions "
      f"({100*top50_reach/total_reach_flag:.0f}% of flagged reach)")

flagged pages by archetype (valued by reach at risk, in millions):
                    n  at_risk_reach  avg_reach  avg_cost_tier
archetype                                                     
fresh_giant       655            9.0    13675.0            1.0
giant             124            2.1    16936.0            1.0
watch             671            0.8     1250.0            0.0
visible_low_ctr   460            0.6     1254.0            2.0
quiet_low_reach  1416            0.1       87.0            0.0
visible_steady    116            0.1     1030.0            2.0

at-risk reach in the flagged set: 12,716,596 impressions
reach covered by reviewing the top-50 rows: 199,778 impressions (2% of flagged reach)


## 3. Human review + the no-go list

**Every row in the queue is a question, not an order.** Before acting on any row, a person checks the five lines below — the reason code + archetype exist to make that check fast, not to replace it.

**What should never be automated.**

- **Never auto-rewrite or auto-publish content.** "Flagged" is an observed association with decline; there is no matched experiment in this snapshot showing refresh *causes* recovery. An automated edit on that evidence is a guess wearing a model's coat.
- **Never auto-delete, consolidate, or redirect** pages. Those destroy reach; they are a human decision with full context.
- **Never auto-send client-facing numbers** from this queue. P@K and value proxies were computed on pseudonyms, not on anyone's live site.
- **Never act on sub-300-impression pages** as actions — that is the noise floor; they are `monitor` only.
- **Never forecast next-quarter impressions** from the queue for a review meeting. The queue orders reviews; it is not a projection.
- **Never claim this queue predicts Google's ranking algorithm**, or that the model "won" by itself — it beat a hand-rule *at ranking held-out pages for review*, measured out-of-sample.

In [4]:
# --- 4. the human review + the no-go list, with the concrete edge cases -------
checklist = [
    "Verify the underlying numbers on the page: impressions_90d, avg_position, CTR, last update.",
    "For 'giant' rows: confirm the trend and the export are real - no hand-check is skipped even for rank 1.",
    "For 'visible_low_ctr': read the actual title/meta before editing - 30 seconds prevents a talent mistake.",
    "For 'mature_stale': open the page, confirm the content is actually stale (not timeless), then plan the refresh.",
    "Below the noise floor: do not touch. Track queue drift instead (Section 4).",
]
print("Fix these before acting on ANY row:")
for x in checklist:
    print(" -", x)

print("\nNo-go (never automated) - each with the honest reason:")
nogo = [
    ("auto-publish refreshes", "no matched experiment shows refresh causes recovery on this snapshot"),
    ("auto-delete/consolidate/redirect", "destroys reach; needs human context"),
    ("auto-send client numbers", "all statistics computed on pseudonymous data"),
    ("act on <300-impression pages", "noise floor"),
    ("forecast next-quarter impressions", "the queue orders reviews; it is not a projection"),
    ("claim Google-algorithm knowledge", "the model ranked held-out pages; that is all it did"),
]
for what, why in nogo:
    print(f" - never {what}: {why}")

print("\nThe edge case the score under-ranks, caught by rule #1 (giants first):")
giant_edges = queue[(queue["impressions_90d"] >= 3000)].sort_values("impressions_90d", ascending=False).head(3)
print(giant_edges[["rank", "proba", "archetype", "action", "impressions_90d", "trend_direction"]].to_string(index=False))

Fix these before acting on ANY row:
 - Verify the underlying numbers on the page: impressions_90d, avg_position, CTR, last update.
 - For 'giant' rows: confirm the trend and the export are real - no hand-check is skipped even for rank 1.
 - For 'visible_low_ctr': read the actual title/meta before editing - 30 seconds prevents a talent mistake.
 - For 'mature_stale': open the page, confirm the content is actually stale (not timeless), then plan the refresh.
 - Below the noise floor: do not touch. Track queue drift instead (Section 4).

No-go (never automated) - each with the honest reason:
 - never auto-publish refreshes: no matched experiment shows refresh causes recovery on this snapshot
 - never auto-delete/consolidate/redirect: destroys reach; needs human context
 - never auto-send client numbers: all statistics computed on pseudonymous data
 - never act on <300-impression pages: noise floor
 - never forecast next-quarter impressions: the queue orders reviews; it is not a projection

## 4. Monitoring / retrain triggers

The model is trained on one snapshot; it will go stale. Light, non-production triggers, printed and stored in the receipt so next quarter has a baseline to compare against.

In [5]:
# --- 5. monitoring / retrain triggers + stored baselines ---------------------
snap_baselines = {
    "n_rows": int(len(df)),
    "median_impressions_90d": float(df["impressions_90d"].median()),
    "median_days_since_last_update": float(df["days_since_last_update"].median()),
    "median_avg_position": float(df[df["avg_position"] > 0]["avg_position"].median()),
    "decline_share": float(y.mean()),
}
print("stored snapshot baselines (import these to compare against any future export):")
for k, v in snap_baselines.items():
    print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")

valid_p10 = precision_at_k(yte, pte)[0]
valid_auc = round(float(roc_auc_score(yte, pte)), 3)
print("\nretrain triggers (light, non-production):")
print(f"  - new export arrives AND grouped-holdout AUC < {valid_auc - 0.03:.3f} (validated now: {valid_auc})")
print(f"  - new export arrives AND grouped-holdout P@10 < {max(0.80, valid_p10 - 0.10):.2f} (validated now: {valid_p10:.2f})")
print("  - any of the three baseline medians shifts by more than 20% vs the stored values above")
print("  - editor feedback: among the top-50 the team actually reviewed, the observed-decline")
print("    confirmation rate falls below the base rate (~51%): re-validate before trusting the queue")

stored snapshot baselines (import these to compare against any future export):
  n_rows: 30000
  median_impressions_90d: 731.00
  median_days_since_last_update: 20.00
  median_avg_position: 11.40
  decline_share: 0.54

retrain triggers (light, non-production):
  - new export arrives AND grouped-holdout AUC < 0.592 (validated now: 0.622)
  - new export arrives AND grouped-holdout P@10 < 0.80 (validated now: 0.90)
  - any of the three baseline medians shifts by more than 20% vs the stored values above
  - editor feedback: among the top-50 the team actually reviewed, the observed-decline
    confirmation rate falls below the base rate (~51%): re-validate before trusting the queue


## 5. Exports for the paper

Files the research paper builds on next week. The queue **CSV** is regenerated by this notebook and lives in `work/outputs/` (kept out of git by the CI leak-guard by design). It carries one **evaluation-only** column, `is_declining_label` (the observed trend flag, pseudonymized); a production queue would drop that column — an editor-facing priority list never shows the answer the ranker is estimating. The **figures** are committed under `work/figures/`; the **metrics receipt** (JSON) is committed under `work/outputs/` so the paper's numbers trace to the executed run.

In [6]:
# --- 6. exports for the paper -----------------------------------------------
queue_csv = OUT / "action_playbook_queue.csv"
export_cols = ["rank", "content_id", "proba", "predicted", "archetype", "action",
               "cost_tier", "reason", "impressions_90d", "clicks_90d", "avg_position",
               "ctr", "days_since_last_update", "content_age_days", "is_declining_label",
               "main_intent", "content_type"]
queue[export_cols].to_csv(queue_csv, index=False)

metrics = {
    "notebook": "w07_action_playbook.ipynb",
    "queue_export": str(queue_csv.relative_to(ROOT)),
    "scope": "client_grouped_holdout_test_pages_only",
    "test_rows": int(len(queue)),
    "test_base_rate": round(float(yte.mean()), 3),
    "validated": {
        "p10": precision_at_k(yte, pte)[0],
        "p50": precision_at_k(yte, pte)[2],
        "roc_auc": valid_auc,
    },
    "flagged_actions": queue.loc[queue["predicted"] == 1, "action"].value_counts().to_dict(),
    "top100_archetypes": queue.head(100)["archetype"].value_counts().to_dict(),
    "reach_at_risk": {
        "flagged_total_impressions": int(queue.loc[queue["predicted"] == 1, "impressions_90d"].sum()),
        "top50_reach_impressions": int(queue.head(50)["impressions_90d"].sum()),
    },
    "snapshot_baselines": snap_baselines,
    "retrain_triggers": {
        "grouped_auc_floor": round(valid_auc - 0.03, 3),
        "p10_floor": max(0.80, valid_p10 - 0.10),
        "drift_trigger_pct": 20,
    },
    "figures": [],
}

# --- figures (reusable in the paper) -----------------------------------------
fig, ax = plt.subplots(figsize=(7, 4))
meth = ["base rate", "week-4 rule", "gradient boosting (validated)"]
k10 = [round(float(yte.mean()), 2), precision_at_k(yte, rule_te)[0], precision_at_k(yte, pte)[0]]
k50 = [round(float(yte.mean()), 2), precision_at_k(yte, rule_te)[2], precision_at_k(yte, pte)[2]]
x = np.arange(3)
ax.bar(x - 0.15, k10, 0.3, label="P@10", color="#4c78a8")
ax.bar(x + 0.15, k50, 0.3, label="P@50", color="#9ecae9")
for xi, v in zip(x - 0.15, k10):
    if v: ax.text(xi, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
for xi, v in zip(x + 0.15, k50):
    if v: ax.text(xi, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(meth)
ax.set_ylim(0, 1.0); ax.set_title("Queue quality on held-out clients (flagging decline)")
ax.legend(frameon=False)
fig.tight_layout()
f1 = FIG / "fig_queue_quality.png"; fig.savefig(f1, dpi=150); plt.close(fig); metrics["figures"].append(str(f1.relative_to(ROOT)))

fig, ax = plt.subplots(figsize=(7, 4))
mix = queue.head(100)["archetype"].value_counts()
ax.barh(mix.index[::-1], mix.values[::-1], color="#4c78a8")
ax.set_title("Archetype mix in the top-100 of the queue")
ax.set_xlabel("pages")
fig.tight_layout()
f2 = FIG / "fig_archetype_mix_top100.png"; fig.savefig(f2, dpi=150); plt.close(fig); metrics["figures"].append(str(f2.relative_to(ROOT)))

def reach_curve(score, desc=True):
    idx = np.argsort(-np.asarray(score)) if desc else np.arange(len(score))
    imp_decl = test["impressions_90d"].values * test["is_declining_label"].values
    total_decl_reach = imp_decl.sum()
    imp_decl_sorted = imp_decl[idx]
    cum = np.cumsum(imp_decl_sorted[:1000]) / total_decl_reach
    return cum

rng = np.random.default_rng(SEED)
rand = rng.permutation(len(test))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(1, 1001), reach_curve(pte), label="gradient boosting", color="#4c78a8")
ax.plot(np.arange(1, 1001), reach_curve(rule_te), label="week-4 rule", color="#f58518")
ax.plot(np.arange(1, 1001), reach_curve(rand), label="random order", color="#bbbbbb", ls="--")
ax.set_xlabel("queue depth (pages reviewed from the top)"); ax.set_ylabel("share of all declining reach captured")
ax.set_title("Reach at risk captured by review order (held-out pages)")
ax.legend(frameon=False)
fig.tight_layout()
f3 = FIG / "fig_reach_curve.png"; fig.savefig(f3, dpi=150); plt.close(fig); metrics["figures"].append(str(f3.relative_to(ROOT)))

metrics_path = OUT / "w07_action_playbook_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2, sort_keys=True))

print("exported:")
for p in [queue_csv, metrics_path] + [FIG / f for f in ["fig_queue_quality.png", "fig_archetype_mix_top100.png", "fig_reach_curve.png"]]:
    print(f"  {p.relative_to(ROOT)}  ({p.stat().st_size:,} bytes)")
print("\nqueue head (exported CSV has all archetype/reason/action columns):")
print(pd.read_csv(queue_csv).head(3)[["rank", "archetype", "action", "cost_tier", "impressions_90d"]].to_string(index=False))
print("\nREADME for the paper: point next week's notebook at work/outputs/action_playbook_queue.csv")
print("and work/figures/*.png; cite numbers from work/outputs/w07_action_playbook_metrics.json.")

exported:
  work/outputs/action_playbook_queue.csv  (1,227,161 bytes)
  work/outputs/w07_action_playbook_metrics.json  (1,187 bytes)
  work/figures/fig_queue_quality.png  (29,661 bytes)
  work/figures/fig_archetype_mix_top100.png  (25,538 bytes)
  work/figures/fig_reach_curve.png  (68,067 bytes)

queue head (exported CSV has all archetype/reason/action columns):
 rank       archetype             action  cost_tier  impressions_90d
    1 visible_low_ctr rewrite_title_meta          2             2846
    2 visible_low_ctr rewrite_title_meta          2             1620
    3 visible_low_ctr rewrite_title_meta          2              502

README for the paper: point next week's notebook at work/outputs/action_playbook_queue.csv
and work/figures/*.png; cite numbers from work/outputs/w07_action_playbook_metrics.json.


## Self-check

Before you submit, confirm each line honestly:

- [x] Ranked queue built from the validated (client-grouped) model, with a deterministic reason code + one action per page
- [x] Archetype -> action mapping table, decay/refresh insight in safe language
- [x] Intended use + explicit limits + cost/value summary
- [x] Human-review rules and the no-go list (nothing auto-publishes, auto-deletes, or forecasts)
- [x] Monitoring/retrain triggers with stored baselines
- [x] Queue CSV exported to work/outputs/, figures to work/figures/, metrics JSON committed
- [x] Notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use observed / measured / directional / decision-support